## Main analysis of the code and generation of results

In [1]:
%load_ext autoreload
%autoreload 2

In [16]:
import numpy as np
import pandas as pd
from data_input import load_excess_returns, prepare_returns
from pathlib import Path
from markowitz import markowitz_unconstrained
from sharpe_ratio import calculate_sharpe_ratio, rolling_sharpe_ratio
from one_over_n import equal_weight_strategy, market_weight_strategy

In [3]:
import bayesian_averaging

In [4]:
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

DATA_PATH = BASE_DIR / "../../datasets/12_Industry_Portfolios_Daily.csv"
RISK_FREE_RATE_PATH = BASE_DIR / "../../datasets/F-F_Research_Data_Factors_daily.csv"

#next steps:
# need to build markowitz constrained.
# after we have the weight functions, we can look at teh oriingal returns files, calculate teh returns achieved by the portfolio
# we use this to calculate the sharpe ratio

In [5]:
df_excess_returns = load_excess_returns(start_date="2015-07-01")
df_excess_returns_prepared = prepare_returns(df_excess_returns, drop_cols=("Date", "Other", "RF"))
burn_in = 10
periods_until_investment = 990

In [6]:
predicted_means_and_covs = bayesian_averaging.run_core(df_excess_returns_prepared, burn_in=burn_in)

Processing time step 100 / 2642...
Processing time step 200 / 2642...
Processing time step 300 / 2642...
Processing time step 400 / 2642...
Processing time step 500 / 2642...
Processing time step 600 / 2642...
Processing time step 700 / 2642...
Processing time step 800 / 2642...
Processing time step 900 / 2642...
Processing time step 1000 / 2642...
Processing time step 1100 / 2642...
Processing time step 1200 / 2642...
Processing time step 1300 / 2642...
Processing time step 1400 / 2642...
Processing time step 1500 / 2642...
Processing time step 1600 / 2642...
Processing time step 1700 / 2642...
Processing time step 1800 / 2642...
Processing time step 1900 / 2642...
Processing time step 2000 / 2642...
Processing time step 2100 / 2642...
Processing time step 2200 / 2642...
Processing time step 2300 / 2642...
Processing time step 2400 / 2642...
Processing time step 2500 / 2642...
Processing time step 2600 / 2642...


In [13]:
bayesian_weights = markowitz_unconstrained(predicted_means_and_covs, df_excess_returns_prepared, burn_in=burn_in, periods_until_investment=periods_until_investment)
bayesian_weights

,NoDur,Durbl,Manuf,Enrgy,Chems,BusEq,Telcm,Utils,Shops,Hlth,Money,Mkt-RF
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2637,-1.880342,5.310993,6.399110,0.326948,9.554227,11.989546,31.940157,1.225046,5.542941,2.562820,23.609149,9.937038
2638,-4.708909,-2.565054,1.052029,-13.609934,13.709365,5.651524,28.023704,-14.430482,3.615086,6.026119,30.988521,6.844395
2639,0.663703,-21.337821,12.876261,-10.836418,6.388119,13.856635,39.923779,0.546275,7.055329,15.195107,6.493370,8.567903
2640,1.746775,-55.443147,22.565631,5.794347,2.117813,38.195449,50.433233,24.191773,10.886431,11.524376,5.917451,21.043183


In [14]:
sharpe_ratio_bayesian = calculate_sharpe_ratio(df_excess_returns_prepared, bayesian_weights)

sharpe_ratio_bayesian   

0.009364407036134139

In [18]:
sharpe_ratio_rolling = rolling_sharpe_ratio(df_excess_returns_prepared, bayesian_weights, window=50)
sharpe_ratio_rolling

0            NaN
1            NaN
2            NaN
3            NaN
4            NaN
          ...   
2637   -0.152226
2638   -0.148612
2639   -0.133979
2640   -0.156175
2641   -0.160206
Length: 2642, dtype: float64

In [19]:
df_excess_Returns_1963 = load_excess_returns(start_date="1963-07-01", end_date="2011-12-31")
df_excess_Returns_1963_prepared = prepare_returns(df_excess_Returns_1963, drop_cols=("Date", "Other", "RF"))

In [20]:
one_over_n_weights = equal_weight_strategy(df_excess_returns_prepared, burn_in=1000)

sharpe_ratio_1_over_n = calculate_sharpe_ratio(df_excess_returns_prepared, one_over_n_weights)
sharpe_ratio_1_over_n


0.030231867016002372

In [21]:
market_weight_strategy = market_weight_strategy(df_excess_returns_prepared, burn_in=1000)
sharpe_ratio_market_weight = calculate_sharpe_ratio(df_excess_returns_prepared, market_weight_strategy)
sharpe_ratio_market_weight

0.03272084470014981

In [ ]:
import json
original_replication_predicted_means_and_covs = bayesian_averaging.run_core(df_excess_Returns_1963_prepared, burn_in=burn_in)
with open(BASE_DIR / "../../datasets/original_replication_predicted_means_and_covs.json", "w") as f:
    json.dump(original_replication_predicted_means_and_covs, f)
original_replication_bayesian_weights = markowitz_unconstrained(
    original_replication_predicted_means_and_covs,
    df_excess_Returns_1963_prepared,
    burn_in=burn_in,
    periods_until_investment=periods_until_investment
)
sharpe_ratio_original_replication = calculate_sharpe_ratio(df_excess_Returns_1963_prepared, original_replication_bayesian_weights)
sharpe_ratio_original_replication

KeyError: 'mu_hat'

In [ ]:
rolling_sharpe_ratio_original_replication = rolling_sharpe_ratio(df_excess_Returns_1963_prepared, original_replication_bayesian_weights, window=50)
rolling_sharpe_ratio_original_replication
